# Train - Hospital Readmission

Trains a Random Forest on `Dataset/processed/hospital_readmission_clean.csv` to predict readmission within 30 days, and saves plots/metrics to `Code/outputs/hospital_readmission/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    average_precision_score, balanced_accuracy_score
)

FIG = "outputs/hospital_readmission"
RES = "outputs/hospital_readmission" 

In [ ]:
df = pd.read_csv("../Dataset/processed/hospital_readmission_clean.csv")

y = df["readmitted_30"]
X = df.drop(columns=["readmitted_30"])
X = pd.get_dummies(X, drop_first=True)
print("Feature matrix after encoding:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=5,
    class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

In [ ]:
report = classification_report(y_test, y_pred, output_dict=True)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))
print("Balanced accuracy:", round(bal_acc, 4))

In [ ]:
with open(f"{RES}/rf_readmission_metrics.json", "w") as f:
    json.dump({
        "classification_report": report,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "balanced_accuracy": bal_acc,
        "n_train": len(X_train), "n_test": len(X_test),
        "n_features": X.shape[1],
        "positive_rate_test": float(y_test.mean()),
    }, f, indent=2)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No <30d readmit", "<30d readmit"],
            yticklabels=["No <30d readmit", "<30d readmit"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest: confusion matrix\n(hospital readmission <30 days)")
plt.tight_layout()
plt.savefig(f"{FIG}/rf_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve: hospital readmission model")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG}/rf_roc_curve.png", dpi=150)
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).nlargest(15)
plt.figure(figsize=(7, 6))
importances.sort_values().plot(kind="barh")
plt.xlabel("Feature importance (Gini)")
plt.title("Random Forest: top 15 features\n(hospital readmission)")
plt.tight_layout()
plt.savefig(f"{FIG}/rf_feature_importance.png", dpi=150)
plt.show()

importances.to_csv(f"{RES}/rf_feature_importance.csv", header=["importance"])
importances.sort_values(ascending=False).head(10)